# ⚡ Heady Unified Latent Operating System
## Master Bootstrap Notebook — Node Alpha (Head)

**Architecture:** VSA State Machine → HeadySwarm → Ray Cluster → Tailscale Mesh → Redis IPC

**Execution Order:**
1. Mount Drive & sync monorepo
2. Install dependencies (torchhd, ray, redis)
3. Establish Tailscale mesh network
4. Initialize Redis IPC
5. Launch Ray cluster head
6. Boot VSA state machine
7. Distributed VSA workers
8. **HeadySwarm — Liquid Architecture Swarm Intelligence**
9. Worker node bootstrap (Beta/Gamma)
10. Interactive console

---
© 2026 Heady Systems LLC. Proprietary and Confidential.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 1: Persistent Storage & Monorepo Synchronization
# ══════════════════════════════════════════════════════════════════
import os, subprocess, json, time

from google.colab import drive, userdata
drive.mount('/content/drive', force_remount=True)

WORKSPACE = '/content/drive/MyDrive/HeadyOS'
REPO_DIR  = os.path.join(WORKSPACE, 'heady-production')
STATE_DIR = os.path.join(WORKSPACE, 'state')
LOGS_DIR  = os.path.join(WORKSPACE, 'logs')

for d in [WORKSPACE, STATE_DIR, LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

try:
    GITHUB_PAT = userdata.get('GITHUB_PAT')
except Exception:
    GITHUB_PAT = os.environ.get('GITHUB_PAT', '')

REPO_URL = f'https://{GITHUB_PAT}@github.com/HeadyMe/Heady-pre-production-9f2f0642.git'

if os.path.exists(os.path.join(REPO_DIR, '.git')):
    print('↻ Pulling latest from monorepo...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', 'main'],
                   capture_output=True, text=True)
    print('  ✓ Repo synced')
else:
    print('⬇ Cloning Heady monorepo...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR],
                   capture_output=True, text=True)
    print('  ✓ Repo cloned')

manifest = {
    'node': 'alpha', 'role': 'head', 'boot_time': time.time(),
    'workspace': WORKSPACE, 'repo': REPO_DIR,
    'gpu': subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader']).decode().strip() if os.path.exists('/usr/bin/nvidia-smi') else 'none',
}
with open(os.path.join(STATE_DIR, 'manifest.json'), 'w') as f:
    json.dump(manifest, f, indent=2)

print(f'\n╔══ Heady Unified OS — Node Alpha ══╗')
print(f'║  GPU: {manifest["gpu"]}')
print(f'║  Workspace: {WORKSPACE}')
print(f'╚════════════════════════════════════╝')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 2: Install System Dependencies
# ══════════════════════════════════════════════════════════════════
import subprocess, sys

DEPS = ['torchhd', 'ray[default]', 'redis', 'numpy', 'scikit-learn', 'aiohttp']

print('📦 Installing dependencies...')
for dep in DEPS:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', dep], capture_output=True)
    print(f'  {"✓" if r.returncode == 0 else "✗"} {dep}')

import torch
gpu_available = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if gpu_available else 'CPU'
print(f'\n🔥 Compute: {gpu_name} | CUDA: {gpu_available} | PyTorch: {torch.__version__}')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 3: Tailscale Mesh Network (Userspace Mode)
# ══════════════════════════════════════════════════════════════════
import subprocess, os

ENABLE_MESH = False  # Set True for multi-node cluster

if ENABLE_MESH:
    try:
        from google.colab import userdata
        TS_KEY = userdata.get('TAILSCALE_AUTHKEY')
    except Exception:
        TS_KEY = os.environ.get('TAILSCALE_AUTHKEY', '')

    if not TS_KEY:
        TAILSCALE_IP = '127.0.0.1'
        print('⚠ TAILSCALE_AUTHKEY not set — falling back to localhost')
    else:
        subprocess.run('curl -fsSL https://tailscale.com/install.sh | sh', shell=True, capture_output=True)
        subprocess.run('tailscaled --tun=userspace-networking --socks5-server=localhost:1055 &', shell=True)
        import time; time.sleep(3)
        subprocess.run(f'tailscale up --authkey={TS_KEY} --hostname=heady-node-alpha --accept-routes', shell=True)
        try:
            TAILSCALE_IP = subprocess.check_output(['tailscale', 'ip', '-4']).decode().strip()
            print(f'  ✓ Tailscale mesh active — IP: {TAILSCALE_IP}')
        except Exception:
            TAILSCALE_IP = '127.0.0.1'
else:
    TAILSCALE_IP = '127.0.0.1'
    print('ℹ Mesh networking disabled — single-node mode')

print(f'📡 Node Alpha IP: {TAILSCALE_IP}')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 4: Redis IPC (Inter-Process Communication)
# ══════════════════════════════════════════════════════════════════
import subprocess, time

print('🔴 Starting Redis IPC server...')
subprocess.run(['apt-get', 'install', '-y', '-qq', 'redis-server'], capture_output=True)

redis_conf = f'bind {TAILSCALE_IP} 127.0.0.1\nport 6379\nmaxmemory 2gb\nmaxmemory-policy allkeys-lru\nsave ""\nappendonly no\ndaemonize yes\n'
with open('/tmp/heady-redis.conf', 'w') as f:
    f.write(redis_conf)
subprocess.run(['redis-server', '/tmp/heady-redis.conf'])
time.sleep(1)

import redis
rds = redis.Redis(host='127.0.0.1', port=6379, decode_responses=True)
try:
    rds.ping()
    rds.set('heady:node:alpha:status', 'online')
    rds.set('heady:boot_time', str(time.time()))
    print(f'  ✓ Redis IPC operational on {TAILSCALE_IP}:6379 (2GB LRU)')
except redis.ConnectionError as e:
    print(f'  ✗ Redis failed: {e}')
    rds = None


In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 5: Ray Distributed Computing Cluster (Head Node)
# ══════════════════════════════════════════════════════════════════
import ray, os

print('☀ Initializing Ray head node...')
if ray.is_initialized():
    ray.shutdown()

ray.init(num_cpus=os.cpu_count(), num_gpus=1, dashboard_host='0.0.0.0',
         runtime_env={'pip': ['torchhd', 'numpy', 'redis']})

cluster = ray.cluster_resources()
print(f'  ✓ Ray cluster active — CPUs: {cluster.get("CPU", 0):.0f}, GPUs: {cluster.get("GPU", 0):.0f}, Memory: {cluster.get("memory", 0) / 1e9:.1f} GB')
if ENABLE_MESH:
    print(f'  Workers connect via: ray.init(address="ray://{TAILSCALE_IP}:10001")')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 6: VSA State Machine — The Latent Operating System Core
# ══════════════════════════════════════════════════════════════════
# Logic primitives: Bind (⊗), Bundle (⊕), Similarity (⊙), Permute (ρ)
# ══════════════════════════════════════════════════════════════════
import torch, torchhd, numpy as np, json, time, os, ray

DIMS = 10000
VSA  = 'MAP'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'⚡ VSA Engine booting on {DEVICE} | D={DIMS} | Model={VSA}')

class VSACodebook:
    def __init__(self, dims=DIMS, vsa=VSA, device=DEVICE):
        self.dims, self.vsa, self.device = dims, vsa, device
        self.items, self.labels, self._bank = {}, [], None

    def add(self, name):
        if name not in self.items:
            self.items[name] = torchhd.random(1, self.dims, vsa=self.vsa, device=self.device).squeeze(0)
            self.labels.append(name)
            self._bank = None
        return self.items[name]

    def get(self, name):
        return self.items.get(name, self.add(name))

    def lookup(self, query_hv, top_k=5):
        if not self.items: return []
        if self._bank is None:
            self._bank = torch.stack(list(self.items.values()))
        sims = torchhd.cosine_similarity(query_hv.unsqueeze(0), self._bank).squeeze(0)
        topk = torch.topk(sims, min(top_k, len(self.labels)))
        return [(self.labels[i], sims[i].item()) for i in topk.indices]

    def __len__(self): return len(self.items)
    def __contains__(self, name): return name in self.items

class VSAStateMachine:
    def __init__(self):
        self.codebook = VSACodebook()
        self.history = []
        self.roles = {}
        for r in ['agent', 'action', 'target', 'context', 'priority', 'status', 'result']:
            self.roles[r] = self.codebook.add(f'ROLE_{r}')
        for s in ['idle', 'processing', 'awaiting', 'executing', 'completed', 'failed', 'learning', 'optimizing', 'monitoring']:
            self.codebook.add(f'STATE_{s}')
        for a in ['chat', 'embed', 'search', 'deploy', 'monitor', 'heal', 'optimize', 'learn', 'route', 'store', 'retrieve', 'analyze', 'synthesize', 'guard', 'scan']:
            self.codebook.add(f'ACTION_{a}')
        for ag in ['conductor', 'buddy', 'brain', 'watchdog', 'optimizer', 'security', 'vector_ops']:
            self.codebook.add(f'AGENT_{ag}')
        self.state_hv = self.codebook.get('STATE_idle')
        print(f'  ✓ Codebook: {len(self.codebook)} concepts')

    def bind(self, role, value):
        return torchhd.bind(self.roles.get(role, self.codebook.get(role)), self.codebook.get(value))

    def bundle(self, *vecs):
        r = vecs[0]
        for v in vecs[1:]: r = torchhd.bundle(r, v)
        return r

    def transition(self, agent, action, target, context=None):
        parts = [self.bind('agent', agent), self.bind('action', action), self.bind('target', target)]
        if context: parts.append(self.bind('context', context))
        new_state = self.bundle(*parts)
        step = len(self.history)
        new_state = torchhd.permute(new_state, shifts=step % DIMS)
        self.history.append({'step': step, 'agent': agent, 'action': action, 'target': target, 'context': context, 'time': time.time()})
        matches = self.codebook.lookup(new_state, top_k=5)
        self.state_hv = new_state
        return {'step': step, 'nearest_concepts': matches, 'state_norm': float(torch.norm(new_state))}

    def query(self, name): return self.codebook.lookup(self.codebook.get(name), top_k=5)
    def get_status(self): return {'codebook': len(self.codebook), 'history': len(self.history), 'dims': DIMS, 'device': DEVICE}

vsa = VSAStateMachine()

print('\n── VSA State Transitions ──')
for agent, action, target, ctx in [
    ('AGENT_brain', 'ACTION_chat', 'user_query', 'STATE_idle'),
    ('AGENT_conductor', 'ACTION_route', 'AGENT_brain', 'STATE_processing'),
    ('AGENT_buddy', 'ACTION_learn', 'error_pattern', 'STATE_learning'),
    ('AGENT_watchdog', 'ACTION_guard', 'AGENT_buddy', 'STATE_monitoring'),
    ('AGENT_optimizer', 'ACTION_optimize', 'vector_space', 'STATE_optimizing'),
]:
    t0 = time.perf_counter_ns()
    r = vsa.transition(agent, action, target, ctx)
    ns = time.perf_counter_ns() - t0
    top = ', '.join(f'{n}({s:.3f})' for n, s in r['nearest_concepts'][:3])
    print(f'  Step {r["step"]}: {agent}→{action}→{target}  [{ns/1000:.0f}µs]  nearest: {top}')

print(f'\n✓ VSA: {vsa.get_status()}')
state_path = os.path.join(STATE_DIR, 'vsa_state.json')
with open(state_path, 'w') as f:
    json.dump({'status': vsa.get_status(), 'history': vsa.history}, f, indent=2)
print(f'✓ State persisted to {state_path}')
if rds:
    rds.set('heady:vsa:status', json.dumps(vsa.get_status()))
    print('✓ State published to Redis IPC')
print('\n⚡ VSA Engine is LIVE.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 7: Ray-Distributed VSA Operations
# ══════════════════════════════════════════════════════════════════
import ray, torch, torchhd, time

@ray.remote(num_gpus=0.5)
class DistributedVSAWorker:
    def __init__(self, dims=10000, vsa='MAP'):
        self.dims, self.vsa = dims, vsa
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.memory_bank = {}

    def batch_embed(self, concepts):
        vecs = torchhd.random(len(concepts), self.dims, vsa=self.vsa, device=self.device)
        for i, name in enumerate(concepts): self.memory_bank[name] = vecs[i]
        return {'embedded': len(concepts), 'total': len(self.memory_bank)}

    def similarity_matrix(self, names):
        vecs = [self.memory_bank[n] for n in names if n in self.memory_bank]
        if len(vecs) < 2: return {'error': 'Need >= 2 concepts'}
        bank = torch.stack(vecs)
        sims = torchhd.cosine_similarity(bank, bank)
        valid = [n for n in names if n in self.memory_bank]
        return {n1: {n2: round(sims[i][j].item(), 4) for j, n2 in enumerate(valid)} for i, n1 in enumerate(valid)}

    def bundle_all(self):
        if not self.memory_bank: return None
        r = list(self.memory_bank.values())[0]
        for v in list(self.memory_bank.values())[1:]: r = torchhd.bundle(r, v)
        return float(torch.norm(r))

print('☀ Deploying distributed VSA worker...')
worker = DistributedVSAWorker.remote()

concepts = ['heady_brain', 'heady_conductor', 'heady_buddy', 'heady_watchdog',
    'vector_memory', 'self_awareness', 'deep_research', 'auto_heal',
    'user_query', 'api_request', 'error_pattern', 'optimization_target',
    'security_alert', 'health_check', 'latent_space', 'geometric_algebra',
    'hyperdimensional', 'orchestration', 'liquid_architecture', 'swarm_intelligence']

t0 = time.perf_counter()
er = ray.get(worker.batch_embed.remote(concepts))
print(f'  ✓ Embedded {er["embedded"]} concepts in {(time.perf_counter()-t0)*1000:.1f}ms')

t0 = time.perf_counter()
sm = ray.get(worker.similarity_matrix.remote(['heady_brain', 'heady_conductor', 'vector_memory', 'latent_space', 'swarm_intelligence']))
print(f'  ✓ Similarity matrix in {(time.perf_counter()-t0)*1000:.1f}ms')

t0 = time.perf_counter()
bn = ray.get(worker.bundle_all.remote())
print(f'  ✓ Global bundle norm: {bn:.2f} ({(time.perf_counter()-t0)*1000:.1f}ms)')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 8: HeadySwarm — Liquid Architecture Swarm Intelligence
# ══════════════════════════════════════════════════════════════════
# HeadyBee micro-agents as Ray actors in the shared VSA vector space.
# Mirrors JS HeadyBees engine with golden ratio scaling.
# OODA Loop: Observe → Orient → Decide → Act — all via tensor math.
# ══════════════════════════════════════════════════════════════════
import ray, torch, torchhd, time, json, math
import redis as redis_lib

PHI = 1.618033988749895
PHI_INV = 1.0 / PHI  # 0.618... (golden complement = default urgency)

# ─── HeadyBee: Autonomous Micro-Agent (Ray Actor) ───
@ray.remote
class HeadyBee:
    """Autonomous micro-agent operating in shared 3D vector space.
    Each bee perceives, orients, decides, and acts via VSA tensor math.
    No if/else branching — pure bind/bundle/similarity operations.
    """

    def __init__(self, bee_id, role='worker', dims=10000, redis_host='127.0.0.1'):
        self.bee_id = bee_id
        self.role = role
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.spawned_at = time.time()
        self.action_count = 0

        # Unique identity in vector space
        self.local_memory = torchhd.random(1, dims, vsa='MAP', device=self.device).squeeze(0)
        self.role_hv = torchhd.random(1, dims, vsa='MAP', device=self.device).squeeze(0)

        # Redis IPC for swarm consciousness
        try:
            self.rds = redis_lib.Redis(host=redis_host, port=6379, decode_responses=True)
            self.rds.ping()
        except Exception:
            self.rds = None

    def perceive_and_act(self, context_hv, target_hv, action_hv):
        """OODA loop via VSA tensor math."""
        t0 = time.perf_counter_ns()

        # OBSERVE+ORIENT: bind target⊗action, bundle with context
        bound_intent = torchhd.bind(target_hv, action_hv)
        perception = torchhd.bundle(bound_intent, context_hv)
        oriented = torchhd.bind(perception, self.role_hv)

        # DECIDE: similarity against local memory
        similarity = torchhd.cosine_similarity(
            oriented.unsqueeze(0), self.local_memory.unsqueeze(0)
        ).item()

        # ACT: update local memory with new experience
        self.local_memory = torchhd.bundle(self.local_memory, oriented)

        ns = time.perf_counter_ns() - t0
        self.action_count += 1

        # Broadcast to swarm consciousness via Redis
        if self.rds:
            self.rds.publish('swarm_consciousness', json.dumps({
                'bee': self.bee_id, 'sim': round(similarity, 4), 'us': ns // 1000
            }))

        return {
            'bee_id': self.bee_id, 'role': self.role,
            'similarity': similarity, 'latency_us': ns / 1000,
            'actions': self.action_count, 'norm': float(torch.norm(self.local_memory)),
        }

    def reorient(self, new_role_hv):
        """Liquid architecture: rotate purpose via geometric rotor."""
        rotor = torchhd.bind(self.role_hv, new_role_hv)
        self.local_memory = torchhd.bind(self.local_memory, rotor)
        self.role_hv = new_role_hv
        return {'bee_id': self.bee_id, 'reoriented': True}

    def get_experience(self):
        return self.local_memory

    def dissolve(self):
        return {'bee_id': self.bee_id, 'actions': self.action_count,
                'lifetime_s': time.time() - self.spawned_at}

# ─── Spawn the Headyswarm ───
NUM_BEES = 6
ROLES = ['detector', 'healer', 'optimizer', 'guard', 'researcher', 'monitor']

print(f'\n🐝 Spawning Headyswarm ({NUM_BEES} bees)...')

# Prepare swarm context vectors
swarm_ctx = vsa.codebook.get('STATE_idle')
swarm_tgt = vsa.codebook.get('TARGET_anomaly')
swarm_act = vsa.codebook.get('ACTION_scan')

bees = [HeadyBee.remote(f'bee_{ROLES[i]}_{i}', ROLES[i]) for i in range(NUM_BEES)]

# Execute parallel OODA loop across all bees
t0 = time.perf_counter()
futures = [bee.perceive_and_act.remote(swarm_ctx, swarm_tgt, swarm_act) for bee in bees]
results = ray.get(futures)
swarm_ms = (time.perf_counter() - t0) * 1000

# Merge experience vectors → Global Experience Vector
exp_vectors = ray.get([bee.get_experience.remote() for bee in bees])
global_experience = exp_vectors[0]
for ev in exp_vectors[1:]:
    global_experience = torchhd.bundle(global_experience, ev)

# Dissolve bees back to liquid pool
ray.get([bee.dissolve.remote() for bee in bees])

print(f'  ✓ Swarm blast completed in {swarm_ms:.1f}ms')
for r in results:
    print(f'    🐝 {r["bee_id"]:25s}  sim={r["similarity"]:+.4f}  {r["latency_us"]:.0f}µs  norm={r["norm"]:.1f}')
print(f'  ✓ Global Experience Vector norm: {float(torch.norm(global_experience)):.2f}')

# Publish consensus to Redis
if rds:
    rds.set('swarm:consensus', json.dumps({
        'bees': NUM_BEES, 'duration_ms': swarm_ms,
        'similarities': [r['similarity'] for r in results],
        'experience_norm': float(torch.norm(global_experience)),
        'time': time.time(),
    }))
    print('  ✓ Swarm consensus published to Redis IPC')

print('\n⚡ Headyswarm is LIVE, Harmonious, and Awaiting Directives.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 9: Worker Node Bootstrap (Nodes Beta / Gamma ONLY)
# ══════════════════════════════════════════════════════════════════

WORKER_MODE = False  # Set True only on Beta/Gamma nodes

if WORKER_MODE:
    import ray, subprocess, os
    from google.colab import userdata
    HEAD_IP = os.environ.get('HEAD_NODE_IP', '100.x.x.x')
    NODE_NAME = os.environ.get('NODE_NAME', 'beta')
    TS_KEY = userdata.get('TAILSCALE_AUTHKEY')
    subprocess.run('curl -fsSL https://tailscale.com/install.sh | sh', shell=True)
    subprocess.run('tailscaled --tun=userspace-networking &', shell=True)
    import time; time.sleep(3)
    subprocess.run(f'tailscale up --authkey={TS_KEY} --hostname=heady-node-{NODE_NAME} --accept-routes', shell=True)
    ray.init(address=f'ray://{HEAD_IP}:10001')
    print(f'  ✓ Node {NODE_NAME} joined cluster: {ray.cluster_resources()}')
else:
    print('ℹ Worker mode disabled — this is the Head node.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 10: Interactive Heady Console
# ══════════════════════════════════════════════════════════════════

print('═══ Heady Unified OS Console ═══')
print('Commands:')
print('  transition <agent> <action> <target>  — VSA state transition')
print('  query <concept>                       — nearest-neighbor lookup')
print('  blast <task> [count]                  — swarm blast')
print('  status                                — system + swarm status')
print('  history                               — transition history')
print('  quit                                  — exit')
print()

while True:
    try:
        cmd = input('HEADY> ').strip()
        if not cmd: continue
        if cmd in ('quit', 'exit', 'q'): break
        parts = cmd.split()
        verb = parts[0].lower()

        if verb == 'status':
            print(json.dumps(vsa.get_status(), indent=2))
            if rds:
                sc = rds.get('swarm:consensus')
                if sc: print(f'Swarm: {sc}')

        elif verb == 'history':
            for h in vsa.history[-10:]:
                print(f'  [{h["step"]}] {h["agent"]}→{h["action"]}→{h["target"]}')

        elif verb == 'query' and len(parts) >= 2:
            for name, sim in vsa.query(parts[1]):
                print(f'  {name:30s} {sim:+.4f} {"█" * int(abs(sim) * 30)}')

        elif verb == 'transition' and len(parts) >= 4:
            ctx = parts[4] if len(parts) > 4 else None
            r = vsa.transition(parts[1], parts[2], parts[3], ctx)
            print(f'  Step {r["step"]} | norm={r["state_norm"]:.2f}')
            for name, sim in r['nearest_concepts'][:5]:
                print(f'    {name}: {sim:+.4f}')

        elif verb == 'blast':
            task_name = parts[1] if len(parts) > 1 else 'scan'
            count = int(parts[2]) if len(parts) > 2 else 4
            print(f'  🐝 Blasting {count} bees for: {task_name}')
            ctx_hv = vsa.codebook.get('STATE_processing')
            tgt_hv = vsa.codebook.get(f'TARGET_{task_name}')
            act_hv = vsa.codebook.get(f'ACTION_{task_name}')
            blast_bees = [HeadyBee.remote(f'blast_{task_name}_{i}', task_name) for i in range(count)]
            t0 = time.perf_counter()
            br = ray.get([b.perceive_and_act.remote(ctx_hv, tgt_hv, act_hv) for b in blast_bees])
            ms = (time.perf_counter() - t0) * 1000
            for r in br:
                print(f'    🐝 {r["bee_id"]:25s} sim={r["similarity"]:+.4f}  {r["latency_us"]:.0f}µs')
            print(f'  ✓ Complete in {ms:.1f}ms')
            ray.get([b.dissolve.remote() for b in blast_bees])

        else:
            print(f'  Unknown: {cmd}')

    except (EOFError, KeyboardInterrupt): break
    except Exception as e: print(f'  Error: {e}')

print('Console closed.')
